# 05. LLaMA3 Block Tutorial | LLaMA3 Block 教程

**难度：** Medium | **环境：** CPU-first | **标签：** `模型结构`, `Transformer`, `LLaMA Block` | **目标人群：** 模型结构学习者

---

## 本节导读

前面的 RMSNorm、SwiGLU、RoPE 和 GQA 分别处理尺度、非线性、位置和上下文；进入模型结构后，它们需要在同一个 Decoder Layer 中协同工作。理解这个组合顺序，才能读懂现代 decoder-only 模型如何逐层更新 hidden state。

本节把它们组织成最小 LLaMA-3 Decoder Layer：两次 Pre-Norm、两条变换分支和两次残差更新。学习后应能沿着 `x → attention update → MLP update` 追踪每一步的输入、输出与形状，并判断改动某个组件会影响哪条路径。

**关键词：** `LLaMA3`, `Transformer Block`, `Decoder Layer`

---

## 前置阅读

**导语：** 进入本节前，先能指出 RMSNorm、RoPE、GQA 和 SwiGLU 在 Decoder Layer 中分别承担什么作用，再把它们按执行顺序组合起来。

- [01. RMSNorm Tutorial | RMSNorm 教程](../02_PyTorch_Algorithms/01_RMSNorm_Tutorial.ipynb)
- [02. SwiGLU Activation | SwiGLU 激活](../02_PyTorch_Algorithms/02_SwiGLU_Activation.ipynb)
- [03. RoPE Tutorial | 旋转位置编码教程](../02_PyTorch_Algorithms/03_RoPE_Tutorial.ipynb)
- [04. Attention MHA GQA | 多头注意力](../02_PyTorch_Algorithms/04_Attention_MHA_GQA.ipynb)


---

### Step 1：Decoder Block 的组成、状态流与层堆叠

Decoder-only 模型先把 token 转成 hidden state，再让多个 Decoder Block 依次更新同一条残差流。单个 Block 包含两类变换：Attention 在 token 之间交换信息，MLP 在每个 token 内重组通道；Norm 控制子层输入尺度，残差连接保留主状态并接收增量更新。

| 组件 | 读取什么 | 写回什么 | 跨多个 Block 的意义 |
|---|---|---|---|
| Pre-Norm | 当前 hidden state | 尺度受控的子层输入 | 防止每层直接接收失控尺度 |
| Attention | 当前 token 与可见上下文 | 上下文更新量 | 逐层扩大和重组 token 间信息 |
| MLP / SwiGLU | 单个 token 的通道 | 通道更新量 | 增加逐 token 的非线性容量 |
| Residual stream | Block 输入与两个更新量 | 下一 Block 的主状态 | 为深层网络保留连续的信息与梯度路径 |

![LLaMA3 Decoder Block 与多层残差主干](../docs/public/02_PyTorch_Algorithms/05_llama_block.svg)


### Step 2：两次 Pre-Norm 残差更新

一个 Block 依次执行两次“归一化 → 子层变换 → 残差相加”。Attention 和 MLP 解决的问题不同，因此不能把两条更新合并成一次无序求和；第二次 Pre-Norm 必须读取已经包含 Attention 更新的状态。

$$
h' = h + \operatorname{Attention}(\operatorname{Norm}_1(h)),\qquad
h_{next}=h' + \operatorname{MLP}(\operatorname{Norm}_2(h')).
$$

| 路径 | 输入状态 | 更新来源 | 需要保持的不变量 |
|---|---|---|---|
| Attention 更新 | `h` | token 间上下文 | 输出形状与 `h` 一致 |
| MLP 更新 | `h'` | token 内通道变换 | 必须读取第一次残差后的状态 |
| Block 输出 | 两次残差后的状态 | 主干与增量共同构成 | 可直接作为下一 Block 输入 |

![两次 Pre-Norm 残差更新](../docs/public/02_PyTorch_Algorithms/05_prenorm_residual_updates.svg)


### Step 3：组件替换、深度稳定性与 Block 契约

架构演进经常替换 Attention、Norm 或 MLP，但 Block 外部仍要维持 hidden state 的形状、dtype 和残差接口。局部组件“能运行”不代表堆叠后稳定：深度增加会放大初始化、残差尺度、归一化位置和梯度传播之间的相互作用。

| 结构改动 | Block 内必须保持 | 堆叠后需要观察 |
|---|---|---|
| MHA → GQA / MLA | Attention 输出仍回到 hidden size | 状态容量、质量和 backend 路径 |
| Dense MLP → MoE | token 输出仍回到原位置与通道 | 路由集中、overflow 与通信 |
| LayerNorm → RMSNorm | Norm 输入输出形状不变 | 激活尺度、梯度尺度与收敛 |
| 残差缩放或初始化变化 | 两次更新顺序不变 | 深层激活增长、梯度衰减或爆炸 |

因此本节题目验证一个 Block 的组合顺序和参数梯度；多层稳定性由 60 的专项基准在固定初始化、数据与训练预算下继续验证。


### Step 4：代码设计与 Decoder Block 组装

题目区不重复实现 RMSNorm、RoPE 或 GQA：`StructureNormStub` 只记录 Pre-Norm 的调用位置，`AttentionUpdateStub` 只提供同形状的上下文更新。学习者实现真实 SwiGLU，并将 Attention 与 MLP 更新依次写回残差主干。

| TODO | 机制责任 | 题目区提供 | 测试证据 |
|---|---|---|---|
| TODO 1 | 定义 SwiGLU 的投影契约 | `LlamaMLP` 类与维度参数 | 三个无 bias 投影的形状正确 |
| TODO 2 | 完成门控通道更新 | `F.silu` 与输入接口 | 门控路径和输出形状正确 |
| TODO 3 | 写回 Attention 残差更新 | `norm_1` 与 `attention_update` | MLP 接收第一次更新后的状态 |
| TODO 4 | 写回 MLP 残差更新 | `norm_2` 与 `mlp` | 输出、调用顺序与梯度连通性正确 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class StructureNormStub(nn.Module):
    """记录 Pre-Norm 调用位置的结构替身。

    它只保留形状与可训练缩放接口，不实现 RMSNorm 的均方根统计；
    RMSNorm 的数值机制由第 01 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return x * self.scale


class AttentionUpdateStub(nn.Module):
    """产生同形状的上下文更新，用于验证 Block 组装。

    不实现 causal mask、RoPE、MHA 或 GQA；这些机制分别由第 03、04 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.proj(x)


class LlamaMLP(nn.Module):
    """保留真实 SwiGLU 门控计算的最小 MLP。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # TODO 1：定义 SwiGLU 的三个无 bias 投影。
        # self.gate_proj = ???  # hidden_size → intermediate_size
        # self.up_proj = ???    # hidden_size → intermediate_size
        # self.down_proj = ???  # intermediate_size → hidden_size
        raise NotImplementedError("请先完成 TODO 1：SwiGLU 投影。")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 2：完成 SwiGLU 门控路径。
        # gate = ???  # gate projection 的 SiLU 结果
        # up = ???    # 另一条升维投影
        # return ???  # 对 gate * up 做 down projection
        raise NotImplementedError("请先完成 TODO 2：SwiGLU 门控更新。")


class LlamaDecoderLayer(nn.Module):
    """组装最小 Pre-Norm Decoder Block，验证两次残差更新顺序。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.norm_1 = StructureNormStub(hidden_size)
        self.attention_update = AttentionUpdateStub(hidden_size)
        self.norm_2 = StructureNormStub(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """返回与输入同形状的 Decoder Block 输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        """
        # TODO 3：写回 Attention 残差更新。
        # attention_delta = ???  # norm_1 后的上下文更新
        # hidden_states = ???    # 将更新加回当前主状态
        raise NotImplementedError("请先完成 TODO 3：Attention 残差更新。")

        # TODO 4：写回 MLP 残差更新。
        # mlp_delta = ???       # 对第一次更新后的状态做 norm_2 与 MLP
        # hidden_states = ???   # 将通道更新加回主状态
        # return ???


In [ ]:
# 测试设计：分别验证投影契约、门控路径、两次残差顺序和参数梯度。
# 每个测试只对应一个机制责任，便于定位未完成或顺序错误的 TODO。


def test_swiglu_projection_contract():
    mlp = LlamaMLP(hidden_size=8, intermediate_size=12)
    assert mlp.gate_proj.in_features == 8 and mlp.gate_proj.out_features == 12
    assert mlp.up_proj.in_features == 8 and mlp.up_proj.out_features == 12
    assert mlp.down_proj.in_features == 12 and mlp.down_proj.out_features == 8
    assert mlp.gate_proj.bias is None and mlp.up_proj.bias is None and mlp.down_proj.bias is None


def test_swiglu_gate_path():
    torch.manual_seed(0)
    mlp = LlamaMLP(hidden_size=4, intermediate_size=6)
    x = torch.randn(2, 3, 4)
    expected = mlp.down_proj(F.silu(mlp.gate_proj(x)) * mlp.up_proj(x))
    assert torch.allclose(mlp(x), expected)


class ConstantUpdate(nn.Module):
    """为残差顺序测试提供可辨识、同形状的固定更新。"""

    def __init__(self, value: float):
        super().__init__()
        self.value = value
        self.seen_input = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.seen_input = x.detach().clone()
        return torch.full_like(x, self.value)


def test_residual_update_order():
    layer = LlamaDecoderLayer(hidden_size=4, intermediate_size=6)
    layer.norm_1 = nn.Identity()
    layer.attention_update = ConstantUpdate(2.0)
    layer.norm_2 = nn.Identity()
    layer.mlp = ConstantUpdate(3.0)
    x = torch.ones(1, 2, 4)

    out = layer(x)
    first_update = x + 2.0
    assert torch.allclose(layer.mlp.seen_input, first_update), "MLP 必须接收第一次残差更新后的状态"
    assert torch.allclose(out, x + 5.0), "两次残差更新必须按 Attention → MLP 的顺序累加"


def test_block_shape_calls_and_gradients():
    torch.manual_seed(1)
    layer = LlamaDecoderLayer(hidden_size=8, intermediate_size=12)
    x = torch.randn(2, 3, 8, requires_grad=True)
    out = layer(x)
    assert out.shape == x.shape
    assert layer.norm_1.calls == 1 and layer.attention_update.calls == 1 and layer.norm_2.calls == 1
    out.sum().backward()
    assert all(param.grad is not None for param in layer.parameters())


def run_tests():
    test_swiglu_projection_contract()
    test_swiglu_gate_path()
    test_residual_update_order()
    test_block_shape_calls_and_gradients()
    print("✅ Decoder Block：投影、门控、残差顺序和梯度测试通过")


try:
    run_tests()
except NotImplementedError:
    print("请先按 TODO 1–4 完成 Block 组装，再运行测试。")
    raise


## 参考代码与解析

### 代码

In [ ]:
class StructureNormStub(nn.Module):
    """记录 Pre-Norm 调用位置的结构替身。

    它只保留形状与可训练缩放接口，不实现 RMSNorm 的均方根统计；
    RMSNorm 的数值机制由第 01 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return x * self.scale


class AttentionUpdateStub(nn.Module):
    """产生同形状的上下文更新，用于验证 Block 组装。

    不实现 causal mask、RoPE、MHA 或 GQA；这些机制分别由第 03、04 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.proj(x)


class LlamaMLP(nn.Module):
    """保留真实 SwiGLU 门控计算的最小 MLP。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # TODO 1：定义三个无 bias 投影。
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 2：先得到 gate / up，再计算 SiLU(gate) * up，最后 down 投影回 hidden_size。
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)


class LlamaDecoderLayer(nn.Module):
    """组装最小 Pre-Norm Decoder Block，验证两次残差更新顺序。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.norm_1 = StructureNormStub(hidden_size)
        self.attention_update = AttentionUpdateStub(hidden_size)
        self.norm_2 = StructureNormStub(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """返回与输入同形状的 Decoder Block 输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        """
        # TODO 3：先计算 Attention 更新，并将其写回原始 hidden_states。
        attention_delta = self.attention_update(self.norm_1(hidden_states))
        hidden_states = hidden_states + attention_delta

        # TODO 4：基于第一次更新后的 hidden_states 计算 MLP 更新并写回。
        mlp_delta = self.mlp(self.norm_2(hidden_states))
        hidden_states = hidden_states + mlp_delta
        return hidden_states


### 解析

本题依次完成 SwiGLU 投影、门控前向，以及 Attention → MLP 的两次 Pre-Norm 残差更新。答案与题目区使用同一套模块和控制流，只补全四个机制点。

**TODO 1：定义 SwiGLU 投影**

- `gate_proj` 与 `up_proj` 都将 `hidden_size` 映射到 `intermediate_size`；`down_proj` 再映射回 `hidden_size`。
- 三个投影均不使用 bias，使 MLP 的输入和输出继续与 Block 的残差接口相容。

**TODO 2：完成门控通道更新**

- 先计算 `SiLU(gate_proj(x)) * up_proj(x)`，再通过 `down_proj` 回到 hidden 维。
- 测试把模块输出与同一公式的直接计算对照，确认门控路径没有被遗漏。

**TODO 3：写回 Attention 残差更新**

- `attention_update(norm_1(hidden_states))` 产生同形状更新，再与原始 `hidden_states` 相加。
- `AttentionUpdateStub` 不实现 causal mask、RoPE、MHA 或 GQA；它只验证更新位置、形状与梯度连通性。

**TODO 4：写回 MLP 残差更新**

- MLP 必须接收第一次更新后的状态 `h`，而不是初始输入 `x`。
- 顺序测试用两种固定更新验证 `x → h → out`，避免“形状正确但更新顺序错误”。

### 可选 CPU 延伸：把一个 Block 接到最小语言模型

下面的代码把已组装的 Decoder Block 接到 Embedding 与 LM Head，作为 CPU 上的最小数据流检查。它用于观察 `token id → hidden state → logits → loss` 是否连通，不替代真实模型训练或 GPU benchmark。

In [ ]:
from typing import Optional
class TinyLlamaLM(nn.Module):
    """
        最小 LLaMA 语言模型，仅用于验证 DecoderLayer 结构正确性。

        结构：Embedding → 1 × DecoderLayer → LM Head
        注意：此实现仅为教学验证用途，非生产级模型。
    """
    def __init__(self, vocab_size: int, hidden_size: int, intermediate_size: int):
        super().__init__()
        # 最小 LM 只保留三层：Embedding -> Decoder Layer -> LM Head
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.decoder = LlamaDecoderLayer(hidden_size, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # 1) 把 token id 映射成 hidden states
        hidden_states = self.embed_tokens(input_ids)
        # 2) 经过前面已经组装好的 LLaMA-3 Decoder Layer
        hidden_states = self.decoder(hidden_states)
        # 3) 投影到词表维度，得到 next-token logits
        logits = self.lm_head(hidden_states)
        return logits


def build_toy_batch(
    batch_size: int = 2,
    seq_len: int = 4,
    vocab_size: int = 16,
    device: torch.device = None,
    ):
    """
    构造一个 toy batch，用于最小训练闭环验证。

    Args:
        batch_size: 批次大小
        seq_len: 序列长度
        vocab_size: 词表大小，生成的 token id 在 [0, vocab_size-1] 范围内
        device: 张量所在设备

    Returns:
        input_ids: [batch_size, seq_len]
        labels: [batch_size, seq_len]
    """
    # 一个极小的 next-token toy batch，用来演示最小训练闭环。
    # input_ids 是模型输入
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    # labels 是 input_ids 的简单右移版本
    labels = torch.roll(input_ids, shifts=-1, dims=1)
    # 将最后一个位置的标签设为 0（避免越界，实际训练中会 mask 掉）
    labels[:, -1] = 0
    return input_ids, labels


def train_one_step(
    model: nn.Module, 
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
    )-> tuple[torch.Tensor, torch.Tensor]:
    """
    执行一步训练。

    Args:
        model: 要训练的模型
        batch: (input_ids, labels)
        optimizer: 优化器

    Returns:
        loss: 当前步的 loss
        logits: 模型输出 logits
    """
    input_ids, labels = batch
    # 前向：得到每个位置上的 next-token logits
    logits = model(input_ids)

    # 预测 t+1 token（因果语言建模）
    shift_logits = logits[:, :-1].reshape(-1, logits.size(-1))
    shift_labels = labels[:, 1:].reshape(-1)

    # 只保留 next-token 监督关系：预测 t+1 的 token
    loss = F.cross_entropy(shift_logits, shift_labels)

    # 标准训练三连：清梯度 -> 反传 -> 更新参数
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss, logits


def run_minimal_validation(
    vocab_size: int = 16,
    hidden_size: int = 32,
    intermediate_size: int = 80,
    batch_size: int = 2,
    seq_len: int = 4,
    num_steps: int = 10,
    lr: float = 1e-3,
    device: Optional[torch.device] = None,
) -> None:
    """
    运行最小训练闭环验证。

    Args:
        vocab_size: 词表大小
        hidden_size: 隐藏层维度
        intermediate_size: SwiGLU 中间层维度
        batch_size: 批次大小
        seq_len: 序列长度
        num_steps: 训练步数
        lr: 学习率
        device: 张量所在设备
    """
    print("=== 最小训练闭环验证 ===")
    print(f"配置: vocab_size={vocab_size}, hidden_size={hidden_size}, "
          f"intermediate_size={intermediate_size}, batch_size={batch_size}, "
          f"seq_len={seq_len}, num_steps={num_steps}")

    torch.manual_seed(42)

    model = TinyLlamaLM(vocab_size, hidden_size, intermediate_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    batch = build_toy_batch(batch_size, seq_len, vocab_size)

    losses = []

    for step in range(num_steps):
        loss, logits = train_one_step(model, batch, optimizer)
        losses.append(float(loss))
        if step % 2 == 0 or step == num_steps - 1:
            print(f"  Step {step+1:2d}: loss = {loss.item():.6f}")

    # 验证
    assert logits.shape == (batch_size, seq_len, vocab_size), \
        f"Expected logits shape {(batch_size, seq_len, vocab_size)}, got {logits.shape}"
    assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"

    # 验证 loss 在下降
    if losses[0] > losses[-1]:
        print(f"✅ Loss 下降: {losses[0]:.6f} → {losses[-1]:.6f}")
    else:
        print(f"⚠️ Loss 未明显下降: {losses[0]:.6f} → {losses[-1]:.6f}")

    print("✅ 最小数据逻辑与训练循环通过\n")

if __name__ == "__main__":
    # 运行验证
    run_minimal_validation()

## 相关阅读

完成最小 Block 组装后，可先对照真实模型实现，再沿着残差主路径与 MoE 扩展理解哪些组件会被替换、哪些接口保持不变。

- [Transformers 中的 LLaMA 模型实现](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- [LLaMA 原论文：LLaMA](https://arxiv.org/abs/2302.13971)
- [Block 与残差路径：架构专题](../topic_discussion/llm_architecture_evolution/06_block_residual_path.md)
- [06. MoE 路由器](../02_PyTorch_Algorithms/06_MoE_Router.ipynb)
- [08. 架构技巧](../02_PyTorch_Algorithms/08_Architecture_Tricks.ipynb)